# MI-Reframed Spilled Energy: Diagnostic Notebook

This notebook demonstrates the MI-reframed extensions to spilled energy:
1. **Conditional Entropy** $H_Q(X_{i+1} | x_{i:1})$ and **MI Proxy**
2. **Excess Surprise** $\varepsilon(x_i) = s(x_i) - H_Q$
3. **MI-Calibrated Spilled Energy** $\Delta E_{\text{cal}} = \Delta E / (1 + \alpha \cdot H_Q)$
4. **Diagnostic Taxonomy**: HALLUCINATION / UNCERTAIN / CORRECT / LUCKY GUESS

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os
import torch
import numpy as np
import logging
import transformers
import datasets
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sns.set_style("whitegrid")
sns.set_palette("husl")

logging.basicConfig(level=logging.WARNING, force=True)
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("httpx").setLevel(logging.WARNING)
transformers.logging.set_verbosity_error()
datasets.logging.set_verbosity_error()

from spilled_energy.generation import generate_answer
from spilled_energy.extraction import extract_exact_answer
from spilled_energy.energy import spilled_energy
from spilled_energy.mi import (
    mi_proxy,
    excess_surprise_sequence,
    calibrated_spilled_energy,
    classify_tokens,
    HALLUCINATION,
    UNCERTAIN,
    CORRECT,
    LUCKY_GUESS,
)

## 1. Load Model & Data

In [ ]:
model_name = "meta-llama/Meta-Llama-3-8B"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {model_name} on {device}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, dtype=torch.bfloat16 if device == "cuda" else torch.float32
).to(device)

dataset = load_dataset("trivia_qa", "rc", split="validation", streaming=True)
sample = next(iter(dataset))

question = sample["question"]
ground_truth_aliases = sample["answer"]["aliases"]
print(f"Question: {question}")
print(f"Ground Truth(s): {ground_truth_aliases}")

## 2. Generate & Extract Answer

In [ ]:
prompt = f"Q: {question}\nA:"

gen_output = generate_answer(
    prompt=prompt,
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100,
    do_sample=False,
    device=device,
)

generated_text = gen_output["text"]
print(f"Generated: {generated_text}")

exact_answer = extract_exact_answer(
    question=question,
    long_answer=generated_text,
    model=model,
    tokenizer=tokenizer,
    device=device,
)
print(f"Extracted: {exact_answer}")

is_correct = any(alias.lower() in exact_answer.lower() for alias in ground_truth_aliases)
print(f"Correct: {is_correct}")

## 3. Compute All Metrics

In [ ]:
# Prepare logits and IDs
logits_tensor = torch.stack(gen_output["scores"], dim=1)  # [1, seq_len, vocab]
sequences = gen_output["sequences"]  # [1, total_len]
input_len = sequences.shape[1] - logits_tensor.shape[1]
generated_ids = sequences[:, input_len:]

logits_list = logits_tensor.cpu().float().numpy().tolist()
ids_list = generated_ids.cpu().numpy().tolist()

# Decode tokens for display
tokens = [tokenizer.decode([t]) for t in ids_list[0]]

# --- Original spilled energy ---
delta, E_margin, E = spilled_energy(logits=logits_list, ids=ids_list, beta=1.0)

# --- Excess surprise ---
epsilon, surprise, h_cond_es = excess_surprise_sequence(
    logits=logits_list, ids=ids_list, beta=1.0
)

# --- Calibrated spilled energy ---
delta_cal, delta_raw, h_cond_cal, mi_proxy_vals = calibrated_spilled_energy(
    logits=logits_list, ids=ids_list, beta=1.0, alpha=1.0
)

print(f"Tokens generated: {len(tokens)}")
print(f"Vocab size: {logits_tensor.shape[-1]}")
print(f"Max possible H_Q: {np.log(logits_tensor.shape[-1]):.2f} nats")

## 4. Per-Token Comparison Table

In [ ]:
# Print a compact per-token comparison
print(f"{'Pos':>4} {'Token':>15} {'ΔE':>8} {'ΔE_cal':>8} {'ε':>8} {'H_Q':>8} {'MI_prx':>8}")
print("-" * 73)

n_tokens = min(len(delta[0]), len(delta_cal[0]), len(epsilon[0]))
for i in range(n_tokens):
    d_raw = float(delta[0][i].item() if hasattr(delta[0][i], 'item') else delta[0][i])
    d_cal = float(delta_cal[0][i])
    eps = float(epsilon[0][i])
    h = float(h_cond_cal[0][i]) if i < len(h_cond_cal[0]) else 0.0
    mi = float(mi_proxy_vals[0][i]) if i < len(mi_proxy_vals[0]) else 0.0
    tok = tokens[i] if i < len(tokens) else "?"
    print(f"{i:4d} {tok:>15} {d_raw:8.3f} {d_cal:8.3f} {eps:8.3f} {h:8.3f} {mi:8.3f}")

## 5. Token-Level Plots: Original vs Calibrated

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

positions = np.arange(n_tokens)

# Raw spilled energy
raw_vals = [float(delta[0][i].item() if hasattr(delta[0][i], 'item') else delta[0][i]) for i in range(n_tokens)]
axes[0].bar(positions, raw_vals, color='steelblue', alpha=0.7)
axes[0].set_ylabel('Raw ΔE')
axes[0].set_title('Spilled Energy: Raw vs Calibrated vs Excess Surprise')

# Calibrated spilled energy
cal_vals = [float(delta_cal[0][i]) for i in range(n_tokens)]
axes[1].bar(positions, cal_vals, color='darkorange', alpha=0.7)
axes[1].set_ylabel('Calibrated ΔE')

# Excess surprise
eps_vals = [float(epsilon[0][i]) for i in range(n_tokens)]
axes[2].bar(positions, eps_vals, color='seagreen', alpha=0.7)
axes[2].set_ylabel('Excess Surprise ε')
axes[2].set_xlabel('Token Position')

# Add token labels
if n_tokens <= 30:
    for ax in axes:
        ax.set_xticks(positions)
        ax.set_xticklabels([tokens[i][:8] for i in range(n_tokens)], rotation=45, ha='right', fontsize=7)

plt.tight_layout()
plt.show()

## 6. Diagnostic Taxonomy: 2D Scatter Plot

The $(|\Delta E|, H_Q)$ space is partitioned into four quadrants:
- **HALLUCINATION**: high $|\Delta E|$, low $H_Q$ — model had info but failed
- **UNCERTAIN**: high $|\Delta E|$, high $H_Q$ — model should say "I don't know"
- **CORRECT**: low $|\Delta E|$, low $H_Q$ — confident and right
- **LUCKY GUESS**: low $|\Delta E|$, high $H_Q$ — right by chance

In [ ]:
# Choose thresholds (adjust based on your data)
tau_delta = np.percentile([abs(float(v.item() if hasattr(v, 'item') else v)) for v in delta[0]], 75)
tau_h = np.percentile([float(v) for v in h_cond_cal[0]], 50)

# Classify tokens
labels = classify_tokens(delta, h_cond_cal, tau_delta, tau_h)

# Prepare data for scatter
h_vals = np.array([float(h_cond_cal[0][i]) for i in range(n_tokens)])
abs_delta_vals = np.array([abs(float(delta[0][i].item() if hasattr(delta[0][i], 'item') else delta[0][i])) for i in range(n_tokens)])

color_map = {
    HALLUCINATION: 'red',
    UNCERTAIN: 'orange',
    CORRECT: 'green',
    LUCKY_GUESS: 'gray',
}
colors = [color_map[labels[0][i]] for i in range(n_tokens)]

fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(h_vals, abs_delta_vals, c=colors, s=40, alpha=0.7, edgecolors='black', linewidths=0.5)

# Quadrant boundaries
ax.axhline(y=tau_delta, color='gray', linestyle='--', alpha=0.5, label=f'τ_δ = {tau_delta:.2f}')
ax.axvline(x=tau_h, color='gray', linestyle='--', alpha=0.5, label=f'τ_h = {tau_h:.2f}')

# Label quadrants
x_lo, x_hi = ax.get_xlim()
y_lo, y_hi = ax.get_ylim()
ax.text(tau_h * 0.3, y_hi * 0.9, 'HALLUCINATION', ha='center', fontsize=10, color='red', fontweight='bold')
ax.text((tau_h + x_hi) / 2, y_hi * 0.9, 'UNCERTAIN', ha='center', fontsize=10, color='orange', fontweight='bold')
ax.text(tau_h * 0.3, y_lo + (tau_delta - y_lo) * 0.3, 'CORRECT', ha='center', fontsize=10, color='green', fontweight='bold')
ax.text((tau_h + x_hi) / 2, y_lo + (tau_delta - y_lo) * 0.3, 'LUCKY GUESS', ha='center', fontsize=10, color='gray', fontweight='bold')

# Annotate some tokens
for i in range(n_tokens):
    if abs_delta_vals[i] > tau_delta:
        ax.annotate(tokens[i].strip()[:10], (h_vals[i], abs_delta_vals[i]),
                    fontsize=6, alpha=0.8, textcoords='offset points', xytext=(5, 3))

ax.set_xlabel('Conditional Entropy $H_Q$', fontsize=12)
ax.set_ylabel('$|\\Delta E|$ (Spilled Energy)', fontsize=12)
ax.set_title('Diagnostic Taxonomy')
ax.legend(loc='upper right')

plt.tight_layout()
plt.show()

## 7. Token Classification Summary

In [ ]:
from collections import Counter

counts = Counter(labels[0][:n_tokens])
print("Token Classification Counts:")
for label, count in sorted(counts.items()):
    pct = 100.0 * count / n_tokens
    print(f"  {label:20s}: {count:3d} ({pct:.1f}%)")

print(f"\nThresholds used: τ_δ = {tau_delta:.4f}, τ_h = {tau_h:.4f}")
print(f"Model correctness: {'CORRECT' if is_correct else 'INCORRECT'}")

## 8. Sanity Checks

Verify key mathematical identities from the paper.

In [ ]:
vocab_size = logits_tensor.shape[-1]
log_v = np.log(vocab_size)

print("=== Sanity Checks ===")
print()

# 1. H_Q in [0, log V]
h_vals_check = [float(v) for v in h_cond_cal[0]]
print(f"H_Q range: [{min(h_vals_check):.4f}, {max(h_vals_check):.4f}]")
print(f"Expected:  [0, {log_v:.4f}]")
assert all(0 <= h <= log_v + 1e-5 for h in h_vals_check), "H_Q out of bounds!"
print("✓ H_Q in [0, log V]")
print()

# 2. MI_proxy in [0, log V]
mi_vals_check = [float(v) for v in mi_proxy_vals[0]]
assert all(0 <= m <= log_v + 1e-5 for m in mi_vals_check), "MI proxy out of bounds!"
print("✓ MI_proxy in [0, log V]")
print()

# 3. |delta_cal| <= |delta_raw| (calibration only shrinks)
for i in range(min(len(delta_cal[0]), len(delta[0]))):
    raw = abs(float(delta[0][i].item() if hasattr(delta[0][i], 'item') else delta[0][i]))
    cal = abs(float(delta_cal[0][i]))
    assert cal <= raw + 1e-5, f"Calibrated > raw at position {i}!"
print("✓ |ΔE_cal| <= |ΔE_raw| at all positions")
print()

# 4. surprise >= 0
s_vals = [float(v) for v in surprise[0]]
assert all(s >= -1e-5 for s in s_vals), "Negative surprise!"
print("✓ surprise >= 0")
print()

# 5. epsilon = surprise - H_Q
for i in range(1, min(len(epsilon[0]), len(surprise[0]), len(h_cond_es[0]))):
    computed = float(surprise[0][i]) - float(h_cond_es[0][i])
    actual = float(epsilon[0][i])
    assert abs(computed - actual) < 1e-4, f"Identity broken at position {i}!"
print("✓ ε = s - H_Q (identity verified)")
print()

print("All sanity checks passed.")

## 9. Calibration Effect: High-MI vs Low-MI Suppression

In [ ]:
# Show suppression factor at different entropy levels
h_range = np.linspace(0, log_v, 100)
alpha = 1.0
suppression = 1.0 / (1.0 + alpha * h_range)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(h_range, suppression, 'b-', linewidth=2)
ax.set_xlabel('Conditional Entropy $H_Q$', fontsize=12)
ax.set_ylabel('Suppression Factor', fontsize=12)
ax.set_title(f'Calibration: $1 / (1 + \\alpha \\cdot H_Q)$,  α={alpha}')

# Mark key points
ax.axvline(x=0, color='green', linestyle=':', alpha=0.5, label='H_Q=0 (deterministic)')
ax.axvline(x=log_v, color='red', linestyle=':', alpha=0.5, label=f'H_Q=log V={log_v:.1f} (uniform)')
ax.annotate(f'Factor ≈ {1/(1+alpha*log_v):.3f}', xy=(log_v, 1/(1+alpha*log_v)),
            xytext=(log_v*0.6, 0.3), fontsize=10,
            arrowprops=dict(arrowstyle='->', color='red'))

ax.legend(fontsize=9)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

print(f"At H_Q=0 (confident): suppression = 1.00 (signal preserved)")
print(f"At H_Q=log V={log_v:.1f} (uniform): suppression = {1/(1+alpha*log_v):.4f} (~{1+alpha*log_v:.1f}x reduction)")